# 02 — Tensors, Weights, and Layers

**Description:** Build the minimal numerical foundations of deep learning with NumPy and PyTorch: tensors, shapes, matrix multiplication, weights, biases, linear layers, and a tiny training loop.
**Level:** Beginner
**Tags:** Deep Learning, NumPy, PyTorch, Tensors, Linear Layers

The previous notebook treated a language model as a system that maps context to next-token probabilities. Now we begin opening the box. Neural networks represent information as arrays of numbers and transform those arrays with learned parameters.

By the end, you will be able to:

- distinguish scalars, vectors, matrices, and higher-dimensional tensors;
- read tensor shapes and predict the shapes of operations;
- compute a linear layer by hand and with PyTorch;
- explain what weights and biases do; and
- train a tiny layer by updating its parameters.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)
torch.manual_seed(7)

## 1. Tensors are containers of numbers

A **tensor** is an array of numbers organized along one or more axes. The word sounds specialized, but familiar objects are tensors:

| Name | Number of axes | Example shape | Possible meaning |
|---|---:|---:|---|
| Scalar | 0 | `()` | one loss value |
| Vector | 1 | `(3,)` | three features for one item |
| Matrix | 2 | `(4, 3)` | four items, three features each |
| 3D tensor | 3 | `(2, 4, 3)` | two batches of four items with three features |

The **shape** tells us the size of every axis. Shape is the first thing to inspect when numerical code is confusing.

In [ ]:
scalar = np.array(3.5)
vector = np.array([1.0, 2.0, 3.0])
matrix = np.array([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
cube = np.zeros((2, 2, 3))

for name, value in [("scalar", scalar), ("vector", vector),
                    ("matrix", matrix), ("cube", cube)]:
    print(f"{name:6s} | ndim={value.ndim} | shape={value.shape} | size={value.size}")

### Your turn: read the shape

Imagine that each row represents a token and each column represents a feature. Before running the cell, predict the shape, number of axes, and total number of values. Then change the data by adding another token.

In [ ]:
token_features = np.array([
    [0.2, 0.8, -0.1, 0.5],
    [0.7, 0.1,  0.3, 0.2],
    [0.4, 0.6,  0.0, 0.9],
])

print("shape:", token_features.shape)
print("axes: ", token_features.ndim)
print("values:", token_features.size)

## 2. Axes give dimensions meaning

A shape is meaningful only when we know what each axis represents. For a matrix shaped `(tokens, features)`, axis 0 moves between tokens and axis 1 moves between features. Indexing can select one number, one row, or one column.

In [ ]:
print("one value:       ", token_features[1, 2])
print("one token vector:", token_features[1])
print("one feature:     ", token_features[:, 2])
print("first two tokens:\n", token_features[:2])

## 3. Elementwise operations preserve structure

Adding two equally shaped tensors combines matching positions. Multiplying by a scalar scales every value. These are **elementwise** operations: they do not mix information between positions.

NumPy also supports **broadcasting**, which stretches a compatible smaller array across a larger one. Adding one feature-offset vector to every token row is a common example.

In [ ]:
x = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
offset = np.array([0.1, 0.2, 0.3])

print("scaled:\n", 2 * x)
print("elementwise square:\n", x * x)
print("broadcast addition:\n", x + offset)
print("result shape:", (x + offset).shape)

## 4. Dot products combine features

A dot product multiplies corresponding values and adds the results:

$$x \cdot w = x_1w_1 + x_2w_2 + \cdots + x_dw_d$$

You can think of `w` as a pattern detector. Features with positive weights raise the score; features with negative weights lower it; zero weights are ignored.

In [ ]:
features = np.array([0.8, 0.2, 0.5])
weights = np.array([1.5, -1.0, 0.4])

products = features * weights
score_by_hand = products.sum()
score_with_dot = features @ weights

print("products:     ", products)
print("sum:          ", score_by_hand)
print("dot product:  ", score_with_dot)

### Your turn: design a detector

The three inputs below mean `[contains_question_mark, contains_number, is_long]`. Choose weights that give a high score to questions but a low score to long statements. There is no single correct answer.

In [ ]:
examples = np.array([
    [1.0, 0.0, 0.0],  # short question
    [0.0, 1.0, 1.0],  # long statement with a number
    [1.0, 1.0, 1.0],  # long question with a number
])

my_weights = np.array([1.0, 0.0, 0.0])  # Edit these three values
scores = examples @ my_weights
scores

## 5. A matrix computes several detectors at once

One weight vector produces one output. Stack several weight vectors as columns and matrix multiplication produces several outputs:

$$y = xW + b$$

If `x` has shape `(input_features,)` and `W` has shape `(input_features, output_features)`, then `y` has shape `(output_features,)`. The inner dimensions must match. This shape rule is worth memorizing:

`(..., input_features) @ (input_features, output_features) → (..., output_features)`

In [ ]:
x = np.array([0.8, 0.2, 0.5])                 # shape: (3,)
W = np.array([[ 1.0, -0.5],                   # shape: (3, 2)
              [ 0.0,  1.5],
              [-1.0,  0.4]])
b = np.array([0.1, -0.2])                     # shape: (2,)
y = x @ W + b                                 # shape: (2,)

print("x shape:", x.shape)
print("W shape:", W.shape)
print("b shape:", b.shape)
print("y shape:", y.shape)
print("output: ", y)

## 6. Batches reuse the same weights

Neural networks usually process several examples together. If `X` has one example per row, the same matrix `W` transforms every row. NumPy broadcasts `b` across the batch.

This is both compact and efficient: a single matrix expression replaces a Python loop over examples.

In [ ]:
X = np.array([
    [0.8, 0.2, 0.5],
    [0.1, 0.9, 0.3],
    [0.6, 0.4, 0.7],
    [0.3, 0.3, 0.2],
])                                               # shape: (4, 3)
Y = X @ W + b                                    # shape: (4, 2)

print("X shape:", X.shape)
print("W shape:", W.shape)
print("Y shape:", Y.shape)
print(Y)

### Shape challenge

Without running code, fill in the missing shapes:

- `A` contains 8 examples with 5 features each: shape = `?`
- `B` maps 5 input features to 12 output features: shape = `?`
- `A @ B`: shape = `?`

Then reveal the answer by running the cell.

In [ ]:
A = np.zeros((8, 5))
B = np.zeros((5, 12))
C = A @ B

print("A:    ", A.shape)
print("B:    ", B.shape)
print("A @ B:", C.shape)

## 7. The same tensors in PyTorch

PyTorch tensors look and behave much like NumPy arrays. The important additions are support for accelerators and **automatic differentiation**, which lets PyTorch compute how every parameter affected an error.

PyTorch defaults to 32-bit floating-point values for decimal tensors, while NumPy commonly creates 64-bit floats. In deep learning, 32-bit values are a common balance of speed and precision.

In [ ]:
x_torch = torch.tensor([0.8, 0.2, 0.5])
W_torch = torch.tensor([[ 1.0, -0.5],
                        [ 0.0,  1.5],
                        [-1.0,  0.4]])
b_torch = torch.tensor([0.1, -0.2])
y_torch = x_torch @ W_torch + b_torch

print("value: ", y_torch)
print("shape: ", y_torch.shape)
print("dtype: ", y_torch.dtype)
print("device:", y_torch.device)

## 8. A linear layer packages weights and bias

`nn.Linear(in_features, out_features)` creates a trainable weight matrix and bias vector. PyTorch stores its weight with shape `(out_features, in_features)`, so its internal expression is $xW^T + b$. This storage convention differs from the NumPy matrix above, where output detectors were columns.

The values begin randomly because identical starting weights would make units learn the same thing.

In [ ]:
layer = nn.Linear(in_features=3, out_features=2)

print(layer)
print("weight shape:", layer.weight.shape)
print("bias shape:  ", layer.bias.shape)
print("parameters:  ", sum(parameter.numel() for parameter in layer.parameters()))
print("output:      ", layer(x_torch))

### Verify the layer by hand

A layer is not magic—it applies the same matrix operation we already used. Compute its output directly from the stored parameters and compare the results. Small floating-point differences are normal, so use `torch.allclose` instead of exact equality.

In [ ]:
output_from_layer = layer(x_torch)
output_by_hand = x_torch @ layer.weight.T + layer.bias

print("layer:   ", output_from_layer)
print("by hand: ", output_by_hand)
print("match:   ", torch.allclose(output_from_layer, output_by_hand))

## 9. Layers compose into networks

A single linear layer can only perform a linear transformation. Neural networks become more expressive by alternating linear layers with **nonlinear activation functions**. ReLU is a simple activation that replaces negative values with zero.

The hidden width is a design choice. Here, three input features become four hidden features and then two outputs: `(3) → (4) → (2)`.

In [ ]:
network = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
)

batch = torch.tensor(X, dtype=torch.float32)
hidden = network[0](batch)
activated = network[1](hidden)
output = network[2](activated)

print("input:    ", batch.shape)
print("hidden:   ", hidden.shape)
print("activated:", activated.shape)
print("output:   ", output.shape)

## 10. Weights learn from error

So far, we chose or randomly initialized the weights. During training, a model:

1. makes predictions with its current weights;
2. measures error using a **loss function**;
3. computes gradients—how the loss changes with each parameter;
4. adjusts parameters to reduce the loss; and
5. repeats.

The next example learns the rule $y = 2x + 1$ from a few points. Mean squared error measures the average squared distance between predictions and targets.

In [ ]:
x_train = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y_train = 2 * x_train + 1

regressor = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(regressor.parameters(), lr=0.1)

print("initial weight:", regressor.weight.item())
print("initial bias:  ", regressor.bias.item())
print("initial loss:  ", loss_fn(regressor(x_train), y_train).item())

### Run the training loop

`loss.backward()` fills each parameter's `.grad` attribute. `optimizer.step()` uses those gradients to update the parameters. `optimizer.zero_grad()` clears old gradients before the next step because PyTorch accumulates them by default.

In [ ]:
loss_history = []

for step in range(60):
    predictions = regressor(x_train)       # forward pass
    loss = loss_fn(predictions, y_train)   # measure error

    optimizer.zero_grad()                  # clear old gradients
    loss.backward()                        # compute new gradients
    optimizer.step()                       # update weight and bias

    loss_history.append(loss.item())

print("learned weight:", regressor.weight.item())
print("learned bias:  ", regressor.bias.item())
print("final loss:    ", loss_history[-1])

### Inspect learning

The loss should fall rapidly as the weight approaches 2 and the bias approaches 1. A logarithmic vertical axis makes the later improvements visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(loss_history, color="#E45756")
axes[0].set(title="Training loss", xlabel="Step", ylabel="Mean squared error", yscale="log")

with torch.no_grad():
    learned_y = regressor(x_train).squeeze().numpy()
axes[1].scatter(x_train.squeeze(), y_train.squeeze(), label="training data")
axes[1].plot(x_train.squeeze(), learned_y, color="#4C78A8", label="learned layer")
axes[1].set(title="What the layer learned", xlabel="x", ylabel="y")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Connect this to language models

Language models use the same ingredients at much larger scale:

- token information is stored in tensors;
- each row often corresponds to one token position;
- learned weight matrices transform features into new features;
- batches add axes for multiple sequences; and
- training adjusts millions or billions of weights to reduce next-token prediction loss.

A transformer contains specialized operations such as attention, but linear layers and matrix multiplications appear throughout it.

## 12. Challenges

1. **Shape reasoning:** Change the first network layer from `nn.Linear(3, 4)` to `nn.Linear(3, 8)`. Which other layer must change, and why?
2. **Parameters:** Calculate the number of parameters in `nn.Linear(3, 4)` by hand. Verify it with `.numel()`. Remember the bias.
3. **Broadcasting:** Give every row of `X` a different scalar offset. What shape should that offset tensor have?
4. **Learning rate:** Retrain the regressor with learning rates `0.01`, `0.1`, and `1.0`. Compare the loss curves. What happens when updates are too small or too large?
5. **New rule:** Change the targets so the layer learns $y = -3x + 0.5$. Check its final weight and bias.
6. **Two inputs:** Create data for $y = 2x_1 - x_2 + 1$ and train `nn.Linear(2, 1)` to recover the rule.

In [ ]:
# Challenge workspace
challenge_layer = nn.Linear(3, 4)
parameter_count = sum(parameter.numel() for parameter in challenge_layer.parameters())
print("weight values:", challenge_layer.weight.numel())
print("bias values:  ", challenge_layer.bias.numel())
print("total:        ", parameter_count)

## Takeaways

- Tensors are numerical arrays; shape describes the size of each axis.
- Matrix multiplication mixes input features to produce output features.
- A linear layer computes $y = xW^T + b$ using learnable weights and biases.
- Batches let the same layer process many examples at once.
- Nonlinear activations between linear layers allow richer transformations.
- Training uses loss, gradients, and repeated parameter updates to learn useful weights.

**Next:** *03 — Tokens and Embeddings* will use a learned matrix to turn discrete token IDs into continuous vectors.